# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Baseline Rule

I will prioritize pages that show a strong opportunity for a content refresh.

The score will increase when a page is:
- older and potentially stale,
- receiving meaningful search impressions,
- getting relatively low CTR.

Pages with stronger refresh signals will receive a higher priority score.

Reason Codes

- 'stale_content' — the page is relatively old and may need refreshing.
- 'low_ctr_visible' — the page gets search visibility but has relatively weak CTR.
- 'refresh_opportunity' — the page has multiple signals suggesting that a refresh may be useful.

Actions

- 'refresh' — highest-priority pages for content review.
- 'monitor' — pages with moderate opportunity.
- 'leave' — pages with weak evidence for a refresh.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
print("\n".join(df.columns.tolist()))

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [3]:
import os
import pandas as pd

# Work on a copy
work = df.copy()

# Use data-driven thresholds
age_cutoff = work["content_age_days"].median()
ctr_cutoff = work["ctr"].median()
impressions_cutoff = work["impressions_90d"].median()

# Signals
work["stale_signal"] = work["content_age_days"] >= age_cutoff
work["low_ctr_signal"] = work["ctr"] < ctr_cutoff
work["visible_signal"] = work["impressions_90d"] >= impressions_cutoff

# Baseline score: 1 point for each refresh signal
work["score"] = (
    work["stale_signal"].astype(int)
    + work["low_ctr_signal"].astype(int)
    + work["visible_signal"].astype(int)
)

# ONE reason code: choose the strongest applicable reason
def get_reason(row):
    if row["stale_signal"] and row["visible_signal"]:
        return "stale_visible_page"
    elif row["low_ctr_signal"] and row["visible_signal"]:
        return "low_ctr_visible_page"
    elif row["stale_signal"]:
        return "stale_content"
    else:
        return "other"

work["reason_code"] = work.apply(get_reason, axis=1)

# Action based on score
def get_action(score):
    if score >= 3:
        return "refresh"
    elif score == 2:
        return "monitor"
    else:
        return "leave"

work["action"] = work["score"].apply(get_action)

# Rank highest priority first
work = work.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

work["rank"] = work.index + 1

# Save the ranked queue
os.makedirs("../../work/outputs", exist_ok=True)

output_cols = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "content_age_days",
    "ctr",
    "impressions_90d"
]

queue = work[output_cols]

queue.to_csv(
    "../../work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue written successfully.")
print("Rows:", len(queue))
print("\nTop 10:")
print(queue.head(10).to_string(index=False))

Queue written successfully.
Rows: 30000

Top 10:
 rank           content_id  score        reason_code  action  content_age_days  ctr  impressions_90d
    1 content_8451fc6f034d      3 stale_visible_page refresh               280 0.03           272144
    2 content_c8e9d6ab9013      3 stale_visible_page refresh               362 0.00           208678
    3 content_0e70a832cb7a      3 stale_visible_page refresh               445 0.04           173450
    4 content_91652435f57a      3 stale_visible_page refresh               257 0.06           159590
    5 content_8b36799b7e44      3 stale_visible_page refresh               299 0.02           141400
    6 content_88d367c507a3      3 stale_visible_page refresh               333 0.04           130932
    7 content_e752a4e03dd3      3 stale_visible_page refresh               287 0.01           130892
    8 content_54baba704595      3 stale_visible_page refresh               286 0.01           130617
    9 content_124763d39ca5      3 stale_vi

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda row: (
        "Strong baseline signal because multiple refresh indicators are present."
        if row["score"] == 3
        else "Moderate evidence; signals support review but are not conclusive."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda row: (
        "The page may already be accurate and current despite its age."
        if row["reason_code"] == "stale_visible_page"
        else "Low CTR may be caused by search intent or position rather than content quality."
    ),
    axis=1
)

print(top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].to_string(index=False))

 rank           content_id  action        reason_code                                                         confidence_note                                      what_would_make_it_wrong
    1 content_8451fc6f034d refresh stale_visible_page Strong baseline signal because multiple refresh indicators are present. The page may already be accurate and current despite its age.
    2 content_c8e9d6ab9013 refresh stale_visible_page Strong baseline signal because multiple refresh indicators are present. The page may already be accurate and current despite its age.
    3 content_0e70a832cb7a refresh stale_visible_page Strong baseline signal because multiple refresh indicators are present. The page may already be accurate and current despite its age.
    4 content_91652435f57a refresh stale_visible_page Strong baseline signal because multiple refresh indicators are present. The page may already be accurate and current despite its age.
    5 content_8b36799b7e44 refresh stale_visible_page Strong

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# Review some lower-priority / potentially weak picks
weak_picks = queue.tail(10).copy()

print("Weak picks:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "content_age_days",
            "ctr",
            "impressions_90d"
        ]
    ].to_string(index=False)
)

print("\nLeakage check:")
print("No product flags or future-window outcome/label fields were used.")
print("The baseline uses only content age, CTR, and 90-day impressions.")

Weak picks:
 rank           content_id  score reason_code action  content_age_days    ctr  impressions_90d
29991 content_dfce82404813      0       other  leave               131  33.33                3
29992 content_eb6e97f88ca2      0       other  leave               144  33.33                3
29993 content_336a0d997474      0       other  leave               182  33.33                3
29994 content_a1c65f070bad      0       other  leave               116  33.33                3
29995 content_b96873ca64c1      0       other  leave               144  50.00                2
29996 content_f26233911f33      0       other  leave               140  50.00                2
29997 content_4e95a8389562      0       other  leave               180  50.00                2
29998 content_006b16e7a2e7      0       other  leave               140 100.00                1
29999 content_cfa4d9f1bf0a      0       other  leave               112 100.00                1
30000 content_4272d3a330a3      0     

Weak Picks Review

The weakest picks are not necessarily bad pages. A low baseline score only means that the selected signals provide limited evidence for an immediate refresh.

Leakage Check

The baseline does not use product flags, future-window outcomes, or label-derived fields. It uses only information available from the observed page/search data: content age, CTR, and 90-day impressions.

One limitation is that these signals can indicate an opportunity but cannot prove that changing the content will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.